In [39]:
import json
import numpy as np
import os
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import userdata


client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
model = 'text-embedding-3-small'

# Load data
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)

texts = [f"{t['title']}. {t['description']}" for t in tickets]
response = client.embeddings.create(input=texts, model=model)
embeddings = np.array([data.embedding for data in response.data])





In [40]:
def search_with_category(query, category_filter=None, top_k=5):
    """Search tickets, optionally filtering by category"""
    # Get query embedding
    response = client.embeddings.create(input=[query], model=model)
    query_emb = np.array([response.data[0].embedding])

    # Calculate similarities
    similarities = cosine_similarity(query_emb, embeddings)[0]

    # Get results with category filter
    results = []
    for idx in np.argsort(similarities)[::-1]:
        ticket = tickets[idx]

        # FILL IN THIS LINE: Skip if category doesn't match filter
        # Hint: if category_filter is set AND ticket category doesn't match, skip
        if category_filter and ticket['category'] != category_filter:
            continue

        results.append((ticket, similarities[idx]))
        if len(results) >= top_k:
            break

    return results

In [41]:
# Test it
print("All categories:")
for ticket, score in search_with_category("login problem"):
    print(f"  {score:.3f} [{ticket['category']}] {ticket['title']}")

print("\nOnly 'Authentication' category:")
for ticket, score in search_with_category("login problem", category_filter="Authentication"):
    print(f"  {score:.3f} [{ticket['category']}] {ticket['title']}")


print("\nOnly 'Email' category:")
for ticket, score in search_with_category("login problem", category_filter="Email"):
    print(f"  {score:.3f} [{ticket['category']}] {ticket['title']}")

All categories:
  0.401 [Authentication] Users unable to log in after password reset
  0.376 [Authentication] Session timeout too short
  0.323 [Email] Email notifications not being delivered
  0.323 [Authentication] SSO authentication broken after upgrade
  0.318 [Authentication] Two-factor authentication codes not working

Only 'Authentication' category:
  0.401 [Authentication] Users unable to log in after password reset
  0.376 [Authentication] Session timeout too short
  0.323 [Authentication] SSO authentication broken after upgrade
  0.318 [Authentication] Two-factor authentication codes not working

Only 'Email' category:
  0.323 [Email] Email notifications not being delivered


In [42]:
# Test it
print("All categories:")
for ticket, score in search_with_category("timeout problem"):
    print(f"  {score:.3f} [{ticket['category']}] {ticket['title']}")


print("\nOnly 'Database' category:")
for ticket, score in search_with_category("login problem", category_filter="Database"):
    print(f"  {score:.3f} [{ticket['category']}] {ticket['title']}")



All categories:
  0.481 [Database] Database connection timeout in production
  0.463 [Authentication] Session timeout too short
  0.379 [API] API rate limiting too aggressive
  0.345 [Performance] Memory leak in background worker process
  0.328 [Real-time] Real-time notifications delayed by several minutes

Only 'Database' category:
  0.268 [Database] Database connection timeout in production
